In [1]:
!pip install selenium
!pip install undetected-chromedriver
!pip install webdriver_manager

import csv
import requests
from bs4 import BeautifulSoup as bs
import random
import time
import pandas as pd
import numpy as np
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys

from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

import undetected_chromedriver as uc
import time

In [2]:
browser = uc.Chrome(version_main=145)

In [3]:
url = "https://listado.mercadolibre.com.mx/consolas-videojuegos/consolas/nintendo/nintendo-3ds_NoIndex_True"

In [4]:
browser.get(url)

In [5]:
browser.implicitly_wait(10)

In [6]:
browser.implicitly_wait(10)

In [7]:
cookies_button = WebDriverWait(browser, 10).until(
    EC.element_to_be_clickable((By.XPATH, '/html/body/div[4]/div/div[1]/div/div[2]/button[1]'))
)
    
# Click the button once it's available
cookies_button.click()

In [8]:
browser.implicitly_wait(10)

In [10]:
cp_button = WebDriverWait(browser, 10).until(
    EC.element_to_be_clickable((By.XPATH, '/html/body/div[5]/div/div/div[2]/div/div/div[2]/button[2]'))
)

cp_button.click()

TimeoutException: Message: 
Stacktrace:
Symbols not available. Dumping unresolved backtrace:
	0xfe7dd3
	0xfe7e14
	0xdf1db0
	0xe3c20a
	0xe3c4ab
	0xe7de32
	0xe5ea84
	0xe7b621
	0xe5e7d6
	0xe30049
	0xe30e04
	0x1246924
	0x1241bf7
	0x125f5a0
	0x1000f58
	0x100891d
	0xff0648
	0xff0812
	0xfda21a
	0x75315d49
	0x775cd83b
	0x775cd7c1


In [11]:
html = browser.page_source

In [12]:
soup = bs(html, 'lxml')
soup

<html lang="es-MX"><head><style id="react-aria-pressable-style">@layer {
  [data-react-aria-pressable] {
    touch-action: pan-x pan-y pinch-zoom;
  }
}</style><meta charset="utf-8"/><link href="https://http2.mlstatic.com" rel="preconnect"/><meta content="width=device-width, initial-scale=1.0, maximum-scale=5.0" name="viewport"/><link as="font" crossorigin="" href="https://http2.mlstatic.com/ui/webfonts/v3.0.0/proxima-nova/proximanova-light.woff2" rel="preload" type="font/woff2"/><link as="font" crossorigin="" href="https://http2.mlstatic.com/ui/webfonts/v3.0.0/proxima-nova/proximanova-regular.woff2" rel="preload" type="font/woff2"/><link as="font" crossorigin="" href="https://http2.mlstatic.com/ui/webfonts/v3.0.0/proxima-nova/proximanova-semibold.woff2" rel="preload" type="font/woff2"/><link as="image" href="https://http2.mlstatic.com/D_NQ_NP_626199-MLA74548978406_022024-O.jpg" rel="preload"/><link as="image" href="https://http2.mlstatic.com/D_NQ_NP_799205-MLA74508030657_022024-O.jpg"

In [13]:
soup.find('h3', {'class':'poly-component__title-wrapper'}).text

'Nintendo 3DS Nintendo 3DS Standard CTR-001 Standard - Bueno (Reacondicionado)'

In [14]:
soup.find('div', {'class':'andes-card poly-card poly-card--grid-card poly-card--xlarge poly-card--CORE andes-card--flat andes-card--primary andes-card--padding-0'})

<div class="andes-card poly-card poly-card--grid-card poly-card--xlarge poly-card--CORE andes-card--flat andes-card--primary andes-card--padding-0" data-andes-card="true" data-andes-card-hierarchy="primary" id="_R_gaqp5ee_"><div class="poly-card__portada"><img alt="Nintendo 3DS Nintendo 3DS Standard CTR-001 Standard - Bueno (Reacondicionado)" aria-hidden="true" class="poly-component__picture" data-id="_R_bp6gaqp5ee_" data-testid="picture" decoding="async" is="n-img" loading="lazy" sizes="280px" src="https://http2.mlstatic.com/D_Q_NP_2X_637133-MLU78258398877_082024-E.webp" srcset="https://http2.mlstatic.com/D_Q_NP_637133-MLU78258398877_082024-L.webp 640w, https://http2.mlstatic.com/D_Q_NP_637133-MLU78258398877_082024-F.webp 1200w, https://http2.mlstatic.com/D_Q_NP_637133-MLU78258398877_082024-E.webp 2x, https://http2.mlstatic.com/D_Q_NP_637133-MLU78258398877_082024-T.webp 160w, https://http2.mlstatic.com/D_Q_NP_637133-MLU78258398877_082024-B.webp 800w, https://http2.mlstatic.com/D_Q_NP_

In [15]:
driver = webdriver.Chrome()

def extraer_y_guardar_csv(url, nombre_archivo):
    driver.get(url)
    
    data = []
    
    items = driver.find_elements(By.CLASS_NAME, "ui-search-layout__item")
    
    # Open CSV file for writing
    with open(nombre_archivo, 'w', newline='', encoding='utf-8') as file:
        writer = csv.DictWriter(file, fieldnames=['Titulo', 'Precio', 'Calificacion', 'Ventas/Info'])
        writer.writeheader()
        
        for item in items:
            try:
                titulo = item.find_element(By.CLASS_NAME, "poly-component__title").text

                precio_entero = item.find_element(By.CLASS_NAME, "andes-money-amount__fraction").text
                try:
                    centavos = item.find_element(By.CLASS_NAME, "andes-money-amount__cents").text
                    precio_final = f"{precio_entero}.{centavos}"
                except:
                    precio_final = precio_entero

                try:
                    calificacion = item.find_element(By.CLASS_NAME, "poly-phrase-label").text
                except:
                    calificacion = "N/A"

                try:
                    ventas = item.find_element(By.XPATH, ".//span[contains(text(), 'vendidos')]").text
                    ventas = ventas.replace("| ", "").strip()
                except:
                    ventas = "0 vendidos"

                row_data = {
                    'Titulo': titulo,
                    'Precio': precio_final,
                    'Calificacion': calificacion,
                    'Ventas/Info': ventas
                }
                
                writer.writerow(row_data)
                data.append(row_data)

            except Exception:
                continue

        print(f"¡Hecho! Archivo guardado como: {nombre_archivo}")
    
    return pd.DataFrame(data)

url_objetivo = "https://listado.mercadolibre.com.mx/consolas-videojuegos/consolas/nintendo/nintendo-3ds_NoIndex_True" 
nombre_archivo = "productos_mercadolibre.csv"

df_final = extraer_y_guardar_csv(url_objetivo, nombre_archivo)

driver.quit()

if df_final is not None and not df_final.empty:
    print(df_final.head()) 
    print(f"\n¡Hecho! Archivo guardado con {len(df_final)} filas.")
else:
    print("No se extrajeron datos.")

¡Hecho! Archivo guardado como: productos_mercadolibre.csv
                                              Titulo     Precio Calificacion  \
0  Nintendo 3DS Nintendo 3DS Standard CTR-001 Sta...   3,599.48          N/A   
1  Nintendo Switch Neon Blue And Neon Red Joycon....   5,735.51          4.9   
2  Nintendo 3DS2DS New Super Mario Bros. 2 color ...   3,689.92          N/A   
3  Nintendo 3DS2DS New Super Mario Bros. 2 color ...   3,489.84          N/A   
4  Consola Nintendo 3ds Ll Edición Especial Love ...  11,000.72          N/A   

       Ventas/Info  
0       0 vendidos  
1  +10mil vendidos  
2       0 vendidos  
3       0 vendidos  
4       0 vendidos  

¡Hecho! Archivo guardado con 48 filas.


In [16]:
print(df_final)

                                               Titulo     Precio  \
0   Nintendo 3DS Nintendo 3DS Standard CTR-001 Sta...   3,599.48   
1   Nintendo Switch Neon Blue And Neon Red Joycon....   5,735.51   
2   Nintendo 3DS2DS New Super Mario Bros. 2 color ...   3,689.92   
3   Nintendo 3DS2DS New Super Mario Bros. 2 color ...   3,489.84   
4   Consola Nintendo 3ds Ll Edición Especial Love ...  11,000.72   
5        Nintendo3ds Xl Negro Negro (Reacondicionado)   5,900.33   
6   Nintendo Switch Oled Zelda Edition Reacondicio...  11,138.56   
7   Nintendo3ds Xl Turquesa Con Negro Celeste (Rea...   5,990.33   
8   Nintendo 3ds Cosmo Black (usada) Ver. Japonesa...   4,899.60   
9   Nintendo 3ds Edición Super Mario 3d Land Cib R...   7,000.67   
10  Nintendo3ds Xl Plata Con Negro Gris (Reacondic...   5,990.33   
11  Nintendo 3ds Ll Pokemon Y - Xerneas Yveltal Bl...   9,499.02   
12  Consola Nintendo 3ds Xl | Rojo 32 Gb Original ...   4,800.06   
13                    Consola Nintendo 3ds Xl En

In [ ]:
driver = webdriver.Chrome()

def extraer_y_guardar_csv(url):
    
    try:
        
        # Find product elements
        productos = driver.find_elements(By.CLASS_NAME, "ui-search-layout__item")
        datos_productos = []
        
        print(f"Extrayendo {len(productos)} productos...")

        for item in productos:
            try:
                titulo = item.find_element(By.CLASS_NAME, "poly-component__title").text

                # Lógica de precio
                precio_entero = item.find_element(By.CLASS_NAME, "andes-money-amount__fraction").text
                try:
                    centavos = item.find_element(By.CLASS_NAME, "andes-money-amount__cents").text
                    precio_final = f"{precio_entero}.{centavos}"
                except:
                    precio_final = precio_entero

                # Lógica de calificación
                try:
                    calificacion = item.find_element(By.CLASS_NAME, "poly-phrase-label").text
                except:
                    calificacion = "N/A"

                try:
                    ventas = item.find_element(By.XPATH, ".//span[contains(text(), 'vendidos')]").text
                    ventas = ventas.replace("| ", "").strip()
                except:
                    ventas = "0 vendidos"

                datos_productos.append({
                    'Titulo': titulo,
                    'Precio': precio_final,
                    'Calificacion': calificacion,
                    'Ventas/Info': ventas
                })

            except Exception as e:
                print(f"Error processing item: {e}")
                continue

        df = pd.DataFrame(datos_productos)
        return df
    
    finally:
        driver.quit()

df_final = extraer_y_guardar_csv(url_objetivo)

url_objetivo = "https://listado.mercadolibre.com.mx/consolas-videojuegos/consolas/nintendo/nintendo-3ds_NoIndex_True" 
extraer_y_guardar_csv(url_objetivo)

driver.quit()

if not df_final.empty:
    print(df_final.head()) # Muestra las primeras 5 filas
    df_final.to_csv("productos_mercadolibre.csv", index=False, encoding='utf-8')
    print(f"\n¡Hecho! Archivo guardado con {len(df_final)} filas.")
else:
    print("No se extrajeron datos.")